<a href="https://colab.research.google.com/github/kdeshone/Restroom-Passes/blob/main/Restroom_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Need to create a program where I can either scan a barcode or NFC tag. Once scanned, log the students name associated with the barcode or NFC tag. It will start a timer to track how long the student was in the restroom. To stop the timer, the student must scan or tap back in. If the student is gone for longer than 5 minutes, they receive a violation. After 2 violations, the student will be marked as Probation. If a student is on probation, then they may still use the restroom but their name will be added to the Deduction list. This list will keep track of how many points will be deducted for while on probation. Being placed on probation is an automatic deduction of 3 points. Going to the restroom while on probation will deduct 2 points. Probationary period lasts one week. Students will have 3 passes to use every 3 school weeks. If students are out to the restroom longer than 5 minutes, they will be put into a list that will populate their name, parent's email address (if applicable), and generate a message that lets the parent's know that they have received a restroom strike for violating the restroom policy and that the next strike would result in being placed on a one week probationary period will will come with the loss of points for the week. The restroom count resets after 3 weeks. at the end of every gradin gperiod, if students have not used all of their passes for the grading period, they will be put on a list to receive extra credit for abiding by the restroom policy. If possible, I would want to have some type of notification pop up on the screen showing what student is out to the restroom in green with their name and the time out, once it hits 5 minutes the timer turns red and slowly flashes until the student returns or 10 minutes has passed, whichever comes first.

### 1. System Configuration Constants

These constants define the rules and thresholds for the restroom policy, making the system easy to configure and maintain.

In [ ]:
from datetime import datetime, timedelta

# --- Policy Configuration ---
VIOLATION_THRESHOLD_MINUTES = 5 # Time student can be out before a violation
PROBATION_THRESHOLD_VIOLATIONS = 2 # Number of violations before probation
PROBATION_PERIOD_WEEKS = 1 # Duration of probation
PASSES_PER_CYCLE = 3 # Number of passes students get
PASS_CYCLE_WEEKS = 3 # How often passes reset

# --- Point Deductions ---
PROBATION_AUTO_DEDUCTION = 3 # Points deducted when placed on probation
RESTROOM_DEDUCTION_ON_PROBATION = 2 # Points deducted for restroom use while on probation

# --- Display Configuration ---
WARNING_COLOR = "red"
NORMAL_COLOR = "green"
FLASH_INTERVAL_SECONDS = 1 # For simulated flashing
MAX_DISPLAY_TIME_MINUTES = 10 # Timer stops flashing after this duration if student hasn't returned

### 2. Student Data Structure

We'll use a `Student` class to encapsulate all relevant information about each student, including their ID, name, contact, and their current status regarding violations, probation, and passes. The `student_id` will be unique and can represent their barcode/NFC tag ID.

In [ ]:
class Student:
    def __init__(self, student_id, name, parent_email=None):
        self.student_id = str(student_id) # Ensure ID is a string for consistent barcode/NFC handling
        self.name = name
        self.parent_email = parent_email
        self.violations = 0 # Number of 5+ minute restroom violations
        self.on_probation = False
        self.probation_end_date = None
        self.points_deducted = 0 # Total points deducted due to probation
        self.passes_used_current_cycle = 0 # Passes used in the current 3-week cycle
        self.last_pass_reset_week = datetime.now().isocalendar()[1] # ISO week number when passes last reset

    def __repr__(self):
        return (f"Student(ID: {self.student_id}, Name: {self.name}, Violations: {self.violations}, "
                f"Probation: {self.on_probation}, Points Deducted: {self.points_deducted}, "
                f"Passes Used: {self.passes_used_current_cycle})")

# A dictionary to store all student objects, indexed by their student_id
students_db = {}

# Example Students (for testing purposes)
students_db['1001'] = Student('1001', 'Alice Wonderland', 'alice.p@example.com')
students_db['1002'] = Student('1002', 'Bob The Builder')
students_db['1003'] = Student('1003', 'Charlie Chaplin', 'charlie.c@example.com')

print("Initial Students Database:")
for student_id, student in students_db.items():
    print(student)


Initial Students Database:
Student(ID: 1001, Name: Alice Wonderland, Violations: 0, Probation: False, Points Deducted: 0, Passes Used: 0)
Student(ID: 1002, Name: Bob The Builder, Violations: 0, Probation: False, Points Deducted: 0, Passes Used: 0)
Student(ID: 1003, Name: Charlie Chaplin, Violations: 0, Probation: False, Points Deducted: 0, Passes Used: 0)


### 3. Restroom Log Data Structure

We'll use a `RestroomVisit` class to keep track of each student's visit, including check-in/out times, calculated duration, and whether a violation was incurred. A global list will store all visit records.

### 8. Pass Management and Extra Credit Logic

Functions to manage pass cycles, reset passes, and identify students for extra credit based on their pass usage.

### 7.1 Probation Management

This section adds functions to check and lift probation for students once their probation period has concluded.

In [ ]:
def check_and_lift_probation():
    """Checks all students and lifts probation if their probation end date has passed."""
    current_time = datetime.now()
    print("-- Checking for probation expiry for all students...")
    students_to_remove_from_deduction = []

    for student_id, student in students_db.items():
        if student.on_probation and student.probation_end_date and current_time > student.probation_end_date:
            student.on_probation = False
            student.probation_end_date = None
            student.violations = 0 # Optionally reset violations count after probation is lifted
            students_to_remove_from_deduction.append(student.student_id)
            print(f"    {student.name} (ID: {student.student_id}) probation has been lifted.")

    for sid in students_to_remove_from_deduction:
        if sid in deduction_list:
            deduction_list.remove(sid)

print("Probation management functions defined. `check_and_lift_probation()` is available.")

Probation management functions defined. `check_and_lift_probation()` is available.


### 7.2 Updated `handle_scan_updated` for Probation Check

Integrating the probation check at the start of every scan to ensure student statuses are up-to-date.

In [ ]:
# Update the main handle_scan_updated function to include probation checks
def handle_scan_updated(student_id):
    """Simulates scanning a student's ID, triggering check-in or check-out with updated logic.
    Includes a check for lifting probation at the beginning of every scan."""
    student_id = str(student_id) # Ensure ID is string

    # First, check and lift probation for all students before processing any new scans
    check_and_lift_probation()

    if student_id not in students_db:
        print(f"SCAN ERROR: Student with ID '{student_id}' not found in database.")
        return

    if student_id in active_restroom_users:
        check_out_student_updated(student_id)
    else:
        # Before checking in, check if student has passes available
        student = students_db[student_id]
        # Simplified check for now. More complex logic (e.g., using a pass) would go here.
        # For now, we'll assume they can use a pass if they are not already on probation
        # or if they are on probation, they can still go but will incur deduction.

        # Also check for pass limit
        current_iso_week = datetime.now().isocalendar()[1]
        if (current_iso_week - student.last_pass_reset_week) >= PASS_CYCLE_WEEKS:
            student.passes_used_current_cycle = 0 # Reset passes if cycle passed
            student.last_pass_reset_week = current_iso_week

        if student.passes_used_current_cycle >= PASSES_PER_CYCLE and not student.on_probation:
            print(f"REJECTED: {student.name} has used all {PASSES_PER_CYCLE} passes this cycle and is not on probation.")
            return

        print(f"INFO: {student.name} has {PASSES_PER_CYCLE - student.passes_used_current_cycle} passes remaining this cycle.")

        check_in_student(student_id) # check_in_student remains the same

print("`handle_scan_updated()` has been updated to include probation lifting checks and a basic pass check before check-in.")

`handle_scan_updated()` has been updated to include probation lifting checks and a basic pass check before check-in.


In [ ]:
def update_pass_cycles():
    """Checks all students and resets their passes if a new cycle has begun."""
    current_iso_week = datetime.now().isocalendar()[1]
    print(f"-- Checking pass cycles for all students (current ISO week: {current_iso_week})...")
    for student_id, student in students_db.items():
        # If enough weeks have passed since the last reset, reset passes
        if (current_iso_week - student.last_pass_reset_week) >= PASS_CYCLE_WEEKS:
            if student.passes_used_current_cycle < PASSES_PER_CYCLE:
                # If student has unused passes, they might be eligible for extra credit
                if student.student_id not in extra_credit_list:
                    extra_credit_list.append(student.student_id)
                    print(f"    {student.name} added to Extra Credit list for unused passes.")

            student.passes_used_current_cycle = 0 # Reset passes
            student.last_pass_reset_week = current_iso_week # Update reset week
            print(f"    Passes for {student.name} reset. New cycle started. Passes used: {student.passes_used_current_cycle}")


def check_for_extra_credit_at_period_end():
    """Identifies students for extra credit at the end of a grading period.
    This function would typically be called periodically, e.g., weekly or at grading period end.
    """
    print("-- Checking for extra credit eligibility at grading period end...")
    for student_id, student in students_db.items():
        # Students who have used less than their allotted passes at the end of a cycle
        # (and whose passes haven't been reset yet for a new cycle)
        # Note: The `update_pass_cycles` function already handles adding to extra_credit_list
        # when a cycle completes. This function serves as an explicit check at grading period boundaries.
        if student.passes_used_current_cycle < PASSES_PER_CYCLE:
            if student.student_id not in extra_credit_list:
                extra_credit_list.append(student.student_id)
                print(f"    {student.name} added to Extra Credit list for abiding by policy.")
    print(f"Extra Credit List: {extra_credit_list}")

print("Pass management and extra credit functions defined. `update_pass_cycles()` and `check_for_extra_credit_at_period_end()` are available.")

Pass management and extra credit functions defined. `update_pass_cycles()` and `check_for_extra_credit_at_period_end()` are available.


### 9. Real-time Display Simulation

This section will implement a function to simulate the real-time display of students currently in the restroom, including timer updates, color changes, and flashing warnings based on the policy constants.

In [ ]:
from IPython.display import display, HTML, clear_output
import time

def update_display(elapsed_time=0):
    """Simulates a real-time display for students currently in the restroom.
    Displays name, time out, and changes color/flashes based on violation thresholds.
    `elapsed_time` is used to simulate flashing by alternating display.
    """
    clear_output(wait=True) # Clear previous output for a "real-time" effect
    print("--- RESTROOM STATUS DISPLAY ---")

    if not active_restroom_users:
        print("No students currently in the restroom.")
        return

    current_time = datetime.now()
    display_html = ""

    for student_id, visit in active_restroom_users.items():
        student = students_db[student_id]
        time_out_delta = current_time - visit.check_in_time
        time_out_minutes = time_out_delta.total_seconds() / 60

        display_color = NORMAL_COLOR
        flashing = False

        if time_out_minutes >= VIOLATION_THRESHOLD_MINUTES:
            display_color = WARNING_COLOR
            # Simulate flashing by alternating visibility based on elapsed_time
            if int(elapsed_time / FLASH_INTERVAL_SECONDS) % 2 == 0:
                flashing = True

        if time_out_minutes >= MAX_DISPLAY_TIME_MINUTES:
            # Stop displaying after MAX_DISPLAY_TIME_MINUTES, even if not checked out
            continue

        if flashing and display_color == WARNING_COLOR: # Only flash when it's a warning color
            if int(time_out_minutes * 60 / FLASH_INTERVAL_SECONDS) % 2 == 0:
                name_display = f"<b style='color:{display_color};'>!!! {student.name.upper()} ({time_out_minutes:.1f} min) !!!</b>"
            else:
                name_display = f"<span style='color:{display_color};'>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;</span>"
        else:
            name_display = f"<span style='color:{display_color};'>{student.name} ({time_out_minutes:.1f} min)</span>"

        display_html += f"<p>{name_display}</p>"

    display(HTML(display_html))
    print("-------------------------------")

print("Real-time display simulation function `update_display()` defined.")

Real-time display simulation function `update_display()` defined.


### 10. Simulation Scenario

Let's simulate a scenario with students checking in and out to observe the system's behavior, including timers, violations, and the real-time display.

In [ ]:
# Resetting for a clean simulation (optional, but good for repeatable tests)
students_db['1001'] = Student('1001', 'Alice Wonderland', 'alice.p@example.com')
students_db['1002'] = Student('1002', 'Bob The Builder', 'bob.b@example.com') # Added email for Bob
students_db['1003'] = Student('1003', 'Charlie Chaplin', 'charlie.c@example.com')

active_restroom_users = {}
restroom_logs = []
strike_list = []
deduction_list = []
extra_credit_list = []

print("--- Starting Simulation ---")

# 1. Alice checks in
handle_scan_updated('1001')
update_display()
time.sleep(1) # Simulate 1 second passing for display updates

# 2. Bob checks in
handle_scan_updated('1002')
update_display()
time.sleep(1) # Simulate 1 second passing

# 3. Simulate passage of time, update display a few times
print("\n--- Simulating 3 minutes pass ---")
for i in range(3 * 60):
    time.sleep(1)
    if i % 10 == 0: # Update display every 10 seconds
        update_display(i)

# 4. Charlie checks in
handle_scan_updated('1003')
update_display()
time.sleep(1) # Simulate 1 second passing

# 5. Alice checks out (before any violation)
print("\n--- Alice checks out ---")
handle_scan_updated('1001')
update_display()
time.sleep(1) # Simulate 1 second passing

# 6. Simulate passage of time, Bob and Charlie go over violation threshold
print(f"\n--- Simulating {VIOLATION_THRESHOLD_MINUTES + 2} minutes pass for Bob and Charlie ---")
for i in range((VIOLATION_THRESHOLD_MINUTES + 2) * 60):
    time.sleep(1)
    if i % 5 == 0: # Update display every 5 seconds, more frequently for flashing
        update_display(i)

# 7. Bob checks out (after violation)
print("\n--- Bob checks out ---")
handle_scan_updated('1002')
update_display()
time.sleep(1) # Simulate 1 second passing

# 8. Simulate more time for Charlie, ensuring flashing and potential MAX_DISPLAY_TIME_MINUTES
print(f"\n--- Simulating further for Charlie, nearing {MAX_DISPLAY_TIME_MINUTES} minutes ---")
for i in range((MAX_DISPLAY_TIME_MINUTES - (VIOLATION_THRESHOLD_MINUTES + 2)) * 60):
    time.sleep(1)
    if i % 5 == 0:
        update_display(i)

# 9. Charlie checks out
print("\n--- Charlie checks out ---")
handle_scan_updated('1003')
update_display()

print("\n--- Simulation Complete ---")
print("\nFinal Restroom Logs:")
for log in restroom_logs:
    print(log)

print("\nFinal Student Statuses:")
for student_id, student in students_db.items():
    print(student)

print("\nStrike List:")
for strike in strike_list:
    print(strike)

print("\nDeduction List (Students on Probation):")
print(deduction_list)

print("\nExtra Credit List:")
print(extra_credit_list)

# Demonstrate pass reset (call update_pass_cycles to check for resets)
print("\n--- Running pass cycle update ---")
update_pass_cycles()
print("\nFinal Student Statuses after pass cycle update:")
for student_id, student in students_db.items():
    print(student)

--- RESTROOM STATUS DISPLAY ---
No students currently in the restroom.

--- Simulation Complete ---

Final Restroom Logs:
Visit(ID: 1001, In: 2026-05-08 19:08:56, Out: 2026-05-08 19:11:59, Duration: 3.1 min, Violation: False)
Visit(ID: 1002, In: 2026-05-08 19:08:57, Out: 2026-05-08 19:19:00, Duration: 10.1 min, Violation: True)
Visit(ID: 1003, In: 2026-05-08 19:11:58, Out: 2026-05-08 19:22:01, Duration: 10.1 min, Violation: True)

Final Student Statuses:
Student(ID: 1001, Name: Alice Wonderland, Violations: 0, Probation: False, Points Deducted: 0, Passes Used: 1)
Student(ID: 1002, Name: Bob The Builder, Violations: 1, Probation: False, Points Deducted: 0, Passes Used: 1)
Student(ID: 1003, Name: Charlie Chaplin, Violations: 1, Probation: False, Points Deducted: 0, Passes Used: 1)

Strike List:
{'student_id': '1002', 'parent_email': 'bob.b@example.com', 'message': 'Dear Parent/Guardian,\n\nThis email is to inform you that your child, Bob The Builder, has received a restroom strike for vi

### 11. Simplified UI Simulation (Text-based)

This section will simulate the interactive UI for students and teachers using `input()` and `print()` statements, focusing on the logic and information display.

In [ ]:
import getpass # For masking password input

def display_landing_page():
    """Displays the initial landing page with date, time, and prompt to scan/tap.
    Returns True if a student ID is entered, False if teacher mode is requested, or None to exit.
    """
    clear_output(wait=True)
    current_time = datetime.now()
    print(f"--- RESTROOM MANAGEMENT SYSTEM ---")
    print(f"Date: {current_time.strftime('%Y-%m-%d')}")
    print(f"Time: {current_time.strftime('%H:%M:%S')}")
    print("----------------------------------")
    update_display() # Show current active users
    print("----------------------------------")
    print("Scan your barcode/tap NFC, or type 'teacher' to access teacher mode, or 'exit' to quit.")
    user_input = input("Enter ID or command: ").strip().lower()

    if user_input == 'exit':
        return None
    elif user_input == 'teacher':
        return False # Indicate teacher mode
    else:
        return user_input # Student ID

def run_student_ui(student_id):
    """Simulates the student UI experience."""
    student = students_db.get(student_id)
    if not student:
        print(f"Student with ID '{student_id}' not found. Please try again.")
        time.sleep(2)
        return

    print(f"\n--- Welcome, {student.name}! ---")
    print(f"Remaining Passes this Cycle: {PASSES_PER_CYCLE - student.passes_used_current_cycle}")
    print(f"Violations: {student.violations}")
    print(f"On Probation: {student.on_probation}")
    if student.on_probation:
        print(f"  Probation ends: {student.probation_end_date.strftime('%Y-%m-%d')}")
        print(f"  Points Deducted: {student.points_deducted}")

    # Check if student is already in restroom
    if student_id in active_restroom_users:
        print(f"You are currently checked IN. Would you like to check OUT? (yes/no)")
        action = input().strip().lower()
        if action == 'yes':
            handle_scan_updated(student_id) # This will check them out
            print("Checked out successfully. See you soon!")
        else:
            print("Returning to landing page.")
        time.sleep(2)
        return

    # If not in restroom, offer to check in
    print("\nWould you like to use a pass to check into the restroom? (yes/no)")
    action = input().strip().lower()

    if action == 'yes':
        if student.passes_used_current_cycle >= PASSES_PER_CYCLE and not student.on_probation:
            print(f"Sorry, {student.name}, you have used all {PASSES_PER_CYCLE} passes this cycle and are not on probation.")
            print("Please see a teacher if this is an emergency.")
            time.sleep(3)
            return

        print("\n-- Disclaimer --")
        print("By proceeding, you agree to the restroom policy: max 5 minutes, 2 violations lead to probation, etc.")
        print("Consequences for violating the policy include strikes, point deductions, and parental notification.")
        print("----------------")
        agree = input("Type 'I agree' to accept and check in: ").strip()
        if agree == 'I agree':
            handle_scan_updated(student_id) # This will check them in
            print("Checked in. Timer started. Please return promptly!")
            # Note: The actual timer visualization is handled by update_display on the landing page
        else:
            print("Disclaimer not accepted. Returning to landing page.")
    else:
        print("No pass used. Returning to landing page.")

    time.sleep(2) # Pause before returning to landing page

def run_teacher_ui():
    """Simulates the teacher UI experience."""
    TEACHER_PASSWORD = "teacher123" # Simple password for simulation
    MAX_LOGIN_ATTEMPTS = 3

    for attempt in range(MAX_LOGIN_ATTEMPTS):
        password = getpass.getpass("Enter teacher password: ")
        if password == TEACHER_PASSWORD:
            print("Access granted. Welcome, Teacher!")
            time.sleep(1)
            # Placeholder for teacher dashboard logic
            print("\n--- Teacher Dashboard (Functionality TBD) ---")
            print("Here you can add/edit/delete students, view class rosters, and activate hall pass dashboard.")
            input("Press Enter to return to landing page...")
            return
        else:
            print(f"Incorrect password. {MAX_LOGIN_ATTEMPTS - 1 - attempt} attempts remaining.")
    print("Too many incorrect attempts. Returning to landing page.")
    time.sleep(2)

# Main UI Loop
while True:
    user_action = display_landing_page()

    if user_action is None: # User typed 'exit'
        print("Exiting application. Goodbye!")
        break
    elif user_action is False: # User typed 'teacher'
        run_teacher_ui()
    else: # User entered a student ID
        run_student_ui(user_action)

print("UI simulation ended.")


--- RESTROOM MANAGEMENT SYSTEM ---
Date: 2026-05-08
Time: 19:36:57
----------------------------------
--- RESTROOM STATUS DISPLAY ---
No students currently in the restroom.
----------------------------------
Scan your barcode/tap NFC, or type 'teacher' to access teacher mode, or 'exit' to quit.


### 12. Enhanced Teacher Dashboard Functionalities

This section implements the logic for the teacher to manage the student database and view the hall pass history.

In [1]:
def teacher_manage_students():
    while True:
        clear_output(wait=True)
        print("--- Student Management ---")
        print("1. Add Student")
        print("2. Edit Student")
        print("3. Delete Student")
        print("4. View All Students")
        print("5. Back to Teacher Menu")

        choice = input("Select an option: ")

        if choice == '1':
            sid = input("Enter new Student ID (Barcode/NFC): ")
            name = input("Enter Student Name: ")
            email = input("Enter Parent Email: ")
            students_db[sid] = Student(sid, name, email)
            print(f"Student {name} added.")
        elif choice == '2':
            sid = input("Enter Student ID to edit: ")
            if sid in students_db:
                students_db[sid].name = input(f"New name ({students_db[sid].name}): ") or students_db[sid].name
                students_db[sid].parent_email = input(f"New email ({students_db[sid].parent_email}): ") or students_db[sid].parent_email
                print("Student updated.")
            else:
                print("Student not found.")
        elif choice == '3':
            sid = input("Enter Student ID to delete: ")
            if sid in students_db:
                confirm = input(f"Are you sure you want to delete {students_db[sid].name}? (yes/no): ")
                if confirm.lower() == 'yes':
                    del students_db[sid]
                    print("Student deleted.")
            else:
                print("Student not found.")
        elif choice == '4':
            for sid, s in students_db.items():
                print(s)
            input("\nPress Enter to continue...")
        elif choice == '5':
            break
        time.sleep(1)

def view_hall_pass_dashboard():
    clear_output(wait=True)
    print("--- Hall Pass Dashboard ---")
    if not restroom_logs:
        print("No visit history recorded yet.")
    else:
        import pandas as pd
        # Converting logs to a DataFrame for better tabular visualization
        data = []
        for log in restroom_logs:
            s = students_db.get(log.student_id)
            data.append({
                "Student": s.name if s else "Unknown",
                "ID": log.student_id,
                "Time In": log.check_in_time.strftime('%H:%M:%S'),
                "Time Out": log.check_out_time.strftime('%H:%M:%S') if log.check_out_time else "Active",
                "Duration": f"{log.duration_minutes:.1f}m",
                "Violation": "Yes" if log.violation_triggered else "No"
            })
        display(pd.DataFrame(data))
    input("\nPress Enter to return...")

Now we update the `run_teacher_ui` to use these new management functions.

In [6]:
import getpass
import time
from IPython.display import clear_output

def run_teacher_ui_updated():
    TEACHER_PASSWORD = "teacher123"
    password = getpass.getpass("Enter teacher password: ")
    if password != TEACHER_PASSWORD:
        print("Access denied.")
        time.sleep(2)
        return

    while True:
        clear_output(wait=True)
        print("--- Teacher Dashboard ---")
        print("1. Manage Student Roster")
        print("2. Activate Hall Pass Dashboard (View Logs)")
        print("3. View Strike List")
        print("4. Exit Teacher Mode")

        choice = input("Select an option: ")

        if choice == '1':
            teacher_manage_students()
        elif choice == '2':
            view_hall_pass_dashboard()
        elif choice == '3':
            clear_output(wait=True)
            print("--- Current Strike List ---")
            for strike in strike_list:
                print(f"- {strike['timestamp'].strftime('%Y-%m-%d %H:%M')}: {strike['student_id']} (Notification sent)")
            input("\nPress Enter to continue...")
        elif choice == '4':
            break

print("Teacher UI updated with management and dashboard capabilities.")

Teacher UI updated with management and dashboard capabilities.


In [7]:
# Run this cell to test the updated Teacher Dashboard
# Use the password: teacher123
run_teacher_ui_updated()

--- Hall Pass Dashboard ---


NameError: name 'restroom_logs' is not defined

### Streamlit Version of Restroom Management System

To host this on GitHub and Streamlit Cloud, you will need two files in your repository:
1. `app.py` (The code below)
2. `requirements.txt` (containing `streamlit` and `pandas`)

Note: In a real deployment, you would typically use a database (like SQLite or Firestore) instead of an in-memory dictionary so that your data persists when the app restarts.

In [9]:
!pip install streamlit -q

import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
import time

# --- Initialization ---
if 'students_db' not in st.session_state:
    st.session_state.students_db = {
        '1001': {'name': 'Alice Wonderland', 'email': 'alice@example.com', 'violations': 0, 'on_probation': False, 'points': 0, 'passes': 0},
        '1002': {'name': 'Bob Builder', 'email': 'bob@example.com', 'violations': 0, 'on_probation': False, 'points': 0, 'passes': 0}
    }
if 'logs' not in st.session_state:
    st.session_state.logs = []
if 'active_users' not in st.session_state:
    st.session_state.active_users = {}

# --- UI Layout ---
st.title("🏫 Restroom Management System")

tab1, tab2 = st.tabs(["Student Portal", "Teacher Dashboard"])

with tab1:
    st.header("Scan to Check-In/Out")
    student_id = st.text_input("Scan Barcode / Enter ID", key="scan_input")

    if st.button("Process Scan"):
        if student_id in st.session_state.students_db:
            if student_id in st.session_state.active_users:
                # Check Out Logic
                start_time = st.session_state.active_users.pop(student_id)
                duration = (datetime.now() - start_time).total_seconds() / 60
                st.session_state.logs.append({"ID": student_id, "In": start_time, "Out": datetime.now(), "Duration": round(duration, 2)})
                st.success(f"Checked out {st.session_state.students_db[student_id]['name']}. Duration: {duration:.1f} mins")
            else:
                # Check In Logic
                st.session_state.active_users[student_id] = datetime.now()
                st.session_state.students_db[student_id]['passes'] += 1
                st.info(f"Checked in {st.session_state.students_db[student_id]['name']}. Timer started!")
        else:
            st.error("Student ID not found.")

    # Real-time Status
    st.subheader("Current Students Out")
    for sid, start in st.session_state.active_users.items():
        elapsed = (datetime.now() - start).total_seconds() / 60
        color = "red" if elapsed > 5 else "green"
        st.markdown(f"<p style='color:{color}; font-size:20px;'>{st.session_state.students_db[sid]['name']} - {elapsed:.1f} mins</p>", unsafe_allow_html=True)

with tab2:
    password = st.sidebar.text_input("Teacher Password", type="password")
    if password == "teacher123":
        st.header("Roster & History")
        if st.session_state.logs:
            st.write("### Visit Logs")
            st.table(pd.DataFrame(st.session_state.logs))

        st.write("### Student Database")
        st.write(st.session_state.students_db)
    else:
        st.warning("Enter correct password on the sidebar to view dashboard.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.5 MB/s eta 0:00:00


2026-05-11 13:38:10.623 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-11 13:38:10.625 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-05-11 13:38:10.626 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-11 13:38:10.627 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-11 13:38:10.628 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-11 13:38:10.629 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

In [ ]:
class RestroomVisit:
    def __init__(self, student_id, check_in_time):
        self.student_id = student_id
        self.check_in_time = check_in_time
        self.check_out_time = None
        self.duration_minutes = 0
        self.violation_triggered = False

    def __repr__(self):
        checkout_str = self.check_out_time.strftime('%Y-%m-%d %H:%M:%S') if self.check_out_time else 'N/A'
        return (f"Visit(ID: {self.student_id}, In: {self.check_in_time.strftime('%Y-%m-%d %H:%M:%S')}, "
                f"Out: {checkout_str}, Duration: {self.duration_minutes:.1f} min, Violation: {self.violation_triggered})")

# A list to store all restroom visit records
restroom_logs = []

print("Restroom Logs initialized as an empty list.")

Restroom Logs initialized as an empty list.


### 4. Active Restroom Users

This dictionary will keep track of students currently out of the classroom in the restroom. The key will be `student_id` and the value will be the `RestroomVisit` object that is currently open.

### 7. Parent Notification and Strike List Management

Functions to generate messages for parents and add students to the strike list when a restroom policy violation occurs.

In [ ]:
def generate_parent_message(student_name, violation_duration):
    """Generates a customized message for parents regarding a restroom violation."""
    message = (
        f"Dear Parent/Guardian,\n\n"
        f"This email is to inform you that your child, {student_name}, has received a restroom strike "
        f"for violating the restroom policy. They were out of the restroom for {violation_duration:.1f} minutes, "
        f"exceeding the {VIOLATION_THRESHOLD_MINUTES}-minute limit.\n\n"
        f"Please be aware that the next strike will result in {student_name} being placed on a one-week "
        f"probationary period, which will include a loss of points for that week.\n\n"
        f"We encourage you to discuss the importance of adhering to school policies with your child.\n\n"
        f"Sincerely,\nThe School Administration"
    )
    return message

def add_to_strike_list(student, visit):
    """Adds a student to the strike list and prepares a parent notification."""
    if not student.parent_email:
        print(f"Warning: No parent email found for {student.name}. Cannot send notification for strike.")
        return

    message = generate_parent_message(student.name, visit.duration_minutes)
    strike_entry = {
        'student_id': student.student_id,
        'parent_email': student.parent_email,
        'message': message,
        'timestamp': datetime.now()
    }
    strike_list.append(strike_entry)
    print(f"    {student.name} added to strike list. Parent notification generated (email to {student.parent_email}).")


# Integrate this into process_restroom_visit
def process_restroom_visit_with_strikes(visit):
    """Processes a completed restroom visit, including strike list and parent notifications."""
    student = students_db[visit.student_id]
    print(f"Processing visit for {student.name} (Duration: {visit.duration_minutes:.1f} min)")

    # Check for violation
    if visit.duration_minutes > VIOLATION_THRESHOLD_MINUTES:
        visit.violation_triggered = True
        student.violations += 1
        print(f"    *** VIOLATION! {student.name} exceeded {VIOLATION_THRESHOLD_MINUTES} minutes. Total violations: {student.violations}")
        add_to_strike_list(student, visit) # Add to strike list here

    # Check for probation
    if not student.on_probation and student.violations >= PROBATION_THRESHOLD_VIOLATIONS:
        student.on_probation = True
        student.probation_end_date = datetime.now() + timedelta(weeks=PROBATION_PERIOD_WEEKS)
        student.points_deducted += PROBATION_AUTO_DEDUCTION
        # Add student ID to deduction_list if not already present
        if student.student_id not in deduction_list:
            deduction_list.append(student.student_id)
        print(f"    !!! {student.name} placed on PROBATION! Auto-deduction of {PROBATION_AUTO_DEDUCTION} points. Probation ends: {student.probation_end_date.strftime('%Y-%m-%d')}")

    # Deduct points if on probation and using restroom (only if already on probation)
    if student.on_probation and student.probation_end_date > datetime.now(): # Ensure probation is active
        # If student was just placed on probation this visit and this is the first deduction for restroom use on probation
        # we might want to avoid double counting if PROBATION_AUTO_DEDUCTION *includes* the first restroom use deduction
        # For now, we'll deduct it. Refinement might be needed.
        student.points_deducted += RESTROOM_DEDUCTION_ON_PROBATION
        print(f"    - {student.name} on probation, {RESTROOM_DEDUCTION_ON_PROBATION} points deducted for restroom use. Total deducted: {student.points_deducted}")

    # Handle passes (simple increment for now)
    current_iso_week = datetime.now().isocalendar()[1]
    # Check if a new pass cycle has begun since the last reset
    if (current_iso_week - student.last_pass_reset_week) >= PASS_CYCLE_WEEKS:
        student.passes_used_current_cycle = 0 # Reset passes
        student.last_pass_reset_week = current_iso_week # Update reset week
        print(f"    Passes for {student.name} reset. New cycle started. Passes used: {student.passes_used_current_cycle}")

    student.passes_used_current_cycle += 1
    print(f"    {student.name} used 1 pass. Total passes used this cycle: {student.passes_used_current_cycle}/{PASSES_PER_CYCLE}")

    print(f"Updated student status: {student}")

# Update the main check_out_student function to use the new processing logic
def check_out_student_updated(student_id):
    """Registers a student as checked out of the restroom and processes the visit with updated logic."""
    if student_id not in active_restroom_users:
        print(f"Error: Student with ID {student_id} is not currently checked in.")
        return

    visit = active_restroom_users.pop(student_id) # Remove from active users
    current_time = datetime.now()
    visit.check_out_time = current_time
    duration = (visit.check_out_time - visit.check_in_time).total_seconds() / 60
    visit.duration_minutes = duration
    restroom_logs.append(visit)

    print(f"<-- {students_db[student_id].name} (ID: {student_id}) checked OUT at {current_time.strftime('%H:%M:%S')}. Duration: {duration:.1f} minutes.")

    process_restroom_visit_with_strikes(visit)

def handle_scan_updated(student_id):
    """Simulates scanning a student's ID, triggering check-in or check-out with updated logic."""
    student_id = str(student_id) # Ensure ID is string
    if student_id not in students_db:
        print(f"SCAN ERROR: Student with ID '{student_id}' not found in database.")
        return

    if student_id in active_restroom_users:
        check_out_student_updated(student_id)
    else:
        check_in_student(student_id) # check_in_student remains the same

print("Parent notification and strike list functions defined. `handle_scan_updated(student_id)` is now the primary scan function.")

Parent notification and strike list functions defined. `handle_scan_updated(student_id)` is now the primary scan function.


In [ ]:
# A dictionary to keep track of students currently out of the classroom
# Key: student_id, Value: RestroomVisit object
active_restroom_users = {}

print("Active Restroom Users initialized as an empty dictionary.")

Active Restroom Users initialized as an empty dictionary.


### 5. Violation and Deduction Lists

These lists will store students who have incurred violations or are on the deduction list.

In [ ]:
# List to store students who have received a restroom strike
# Each entry could be a dict: {'student_id': 'ID', 'parent_email': 'email', 'message': 'generated_message'}
strike_list = []

# List to store students currently on the deduction list (i.e., on probation)
# Each entry would likely just be the student_id or the Student object itself
deduction_list = []

# List to store students for extra credit at the end of the grading period
extra_credit_list = []

print("Strike, Deduction, and Extra Credit lists initialized as empty lists.")

Strike, Deduction, and Extra Credit lists initialized as empty lists.


### 6. Core Check-in/Check-out Functions

These functions will handle the primary logic for a student scanning their ID, either checking them into or out of the restroom system.

In [ ]:
def check_in_student(student_id):
    """Registers a student as checked into the restroom."""
    if student_id not in students_db:
        print(f"Error: Student with ID {student_id} not found.")
        return

    if student_id in active_restroom_users:
        print(f"Student {students_db[student_id].name} is already checked out (active_restroom_users contains entry). This shouldn't happen.")
        return

    current_time = datetime.now()
    new_visit = RestroomVisit(student_id, current_time)
    active_restroom_users[student_id] = new_visit
    print(f"--> {students_db[student_id].name} (ID: {student_id}) checked IN at {current_time.strftime('%H:%M:%S')}.")

def check_out_student(student_id):
    """Registers a student as checked out of the restroom and processes the visit."""
    if student_id not in active_restroom_users:
        print(f"Error: Student with ID {student_id} is not currently checked in.")
        return

    visit = active_restroom_users.pop(student_id) # Remove from active users
    current_time = datetime.now()
    visit.check_out_time = current_time
    duration = (visit.check_out_time - visit.check_in_time).total_seconds() / 60
    visit.duration_minutes = duration
    restroom_logs.append(visit)

    print(f"<-- {students_db[student_id].name} (ID: {student_id}) checked OUT at {current_time.strftime('%H:%M:%S')}. Duration: {duration:.1f} minutes.")

    # Further processing for violations, probation, passes will go here later
    process_restroom_visit(visit)

def process_restroom_visit(visit):
    """Processes a completed restroom visit for violations, probation, and passes."""
    student = students_db[visit.student_id]
    print(f"Processing visit for {student.name} (Duration: {visit.duration_minutes:.1f} min)")

    # Check for violation
    if visit.duration_minutes > VIOLATION_THRESHOLD_MINUTES:
        visit.violation_triggered = True
        student.violations += 1
        print(f"    *** VIOLATION! {student.name} exceeded {VIOLATION_THRESHOLD_MINUTES} minutes. Total violations: {student.violations}")
        # Add to strike list logic will go here

    # Check for probation
    if not student.on_probation and student.violations >= PROBATION_THRESHOLD_VIOLATIONS:
        student.on_probation = True
        student.probation_end_date = datetime.now() + timedelta(weeks=PROBATION_PERIOD_WEEKS)
        student.points_deducted += PROBATION_AUTO_DEDUCTION
        deduction_list.append(student.student_id)
        print(f"    !!! {student.name} placed on PROBATION! Auto-deduction of {PROBATION_AUTO_DEDUCTION} points. Probation ends: {student.probation_end_date.strftime('%Y-%m-%d')}")

    # Deduct points if on probation and using restroom
    if student.on_probation and visit.duration_minutes > 0: # Assuming any use while on probation deducts points
        # Only deduct if not already auto-deducted for being placed on probation *for this visit*.
        # This logic might need refinement if 'restroom use while on probation' means *every* use.
        # For now, let's assume it means a separate deduction for using the restroom.
        student.points_deducted += RESTROOM_DEDUCTION_ON_PROBATION
        print(f"    - {student.name} on probation, {RESTROOM_DEDUCTION_ON_PROBATION} points deducted for restroom use. Total deducted: {student.points_deducted}")

    # Handle passes (simple increment for now)
    current_iso_week = datetime.now().isocalendar()[1]
    if current_iso_week - student.last_pass_reset_week >= PASS_CYCLE_WEEKS:
        # Reset passes if a new cycle has begun
        student.passes_used_current_cycle = 0
        student.last_pass_reset_week = current_iso_week
        print(f"    Passes for {student.name} reset. New cycle started.")

    student.passes_used_current_cycle += 1
    print(f"    {student.name} used 1 pass. Total passes used this cycle: {student.passes_used_current_cycle}/{PASSES_PER_CYCLE}")

    print(f"Updated student status: {student}")

def handle_scan(student_id):
    """Simulates scanning a student's ID, triggering check-in or check-out."""
    student_id = str(student_id) # Ensure ID is string
    if student_id not in students_db:
        print(f"SCAN ERROR: Student with ID '{student_id}' not found in database.")
        return

    if student_id in active_restroom_users:
        check_out_student(student_id)
    else:
        check_in_student(student_id)


print("Core check-in/check-out functions defined. `handle_scan(student_id)` is ready to use.")


Core check-in/check-out functions defined. `handle_scan(student_id)` is ready to use.
